In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib numpy qiskit-addon-cutting
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Pagputol ng wire para sa pagtatantya ng expectation values

import TutorialFeedback from '@site/src/components/TutorialFeedback';





*Tantiya sa paggamit: 22 segundo sa Heron processor (PAALALA: Ito ay tantiya lamang. Maaaring mag-iba ang iyong runtime.)*
## Mga resulta ng pag-aaral
Pagkatapos ng tutorial na ito, dapat maunawaan ng mga user ang:
- Paano gamitin ang [`qiskit-addon-cutting`](https://github.com/Qiskit/qiskit-addon-cutting) upang hatiin ang isang malaking circuit sa mas maliliit na subcircuits, na nagbabawas ng epekto ng noise

## Mga Paunang Kaalaman
Iminumungkahi naming maging pamilyar ang mga user sa sumusunod na paksa bago dumaan sa tutorial na ito:
- Paggamit ng [Sampler](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/sampler-v2) primitive, na ginagamit sa workflow na ito

## Background
Ang circuit-knitting ay isang pangkalahatang termino na sumasaklaw sa iba't ibang pamamaraan ng paghahati ng circuit sa maraming mas maliit na subcircuits na may mas kaunting gates o qubits. Ang bawat isa sa mga subcircuits ay maaaring isagawa nang independiyente, at ang panghuling resulta ay nakukuha sa pamamagitan ng ilang classical post-processing sa kinalabasan ng bawat subcircuit. Ang pamamaraang ito ay maaaring ma-access sa [circuit cutting Qiskit addon](https://qiskit.github.io/qiskit-addon-cutting/index.html); tingnan ang [dokumentasyon](https://qiskit.github.io/qiskit-addon-cutting/explanation/index.html) kasama ng iba pang [introductory material](https://qiskit.github.io/qiskit-addon-cutting/tutorials/index.html) para sa detalyadong paliwanag ng pamamaraan.

Ang tutorial na ito ay nakatuon sa isang pamamaraang tinatawag na **wire cutting**, kung saan ang circuit ay pinaghahati sa kahabaan ng wire [\[1\], \[2\]](#references). Tandaan na ang paghahati ay simple sa mga classical circuits dahil ang kinalabasan sa punto ng partition ay maaaring matukoy nang deterministiko, at ito ay 0 o 1 lamang. Gayunpaman, ang estado ng qubit sa punto ng pagputol ay, sa pangkalahatan, isang mixed state. Samakatuwid, ang bawat subcircuit ay kailangang sukatin nang maraming beses sa iba't ibang bases (karaniwang isang tomographically complete na basis, tulad ng Pauli basis [\[3\], \[4\]](#references)) at naaangkop na inihanda sa kanyang eigenstate. Ang figure sa ibaba (pasasalamat: [\[7\]](#references)) ay nagpapakita ng halimbawa ng wire cutting para sa apat na qubit na GHZ state sa tatlong subcircuits. Dito ang $M_j$ ay tumutukoy sa isang set ng mga bases (karaniwang Pauli X, Y at Z), at ang $P_i$ ay tumutukoy sa isang set ng eigenstates (karaniwang $|0\rangle$, $|1\rangle$, $|+\rangle$ at $|+i\rangle$).

![wc-1.png](../docs/images/tutorials/wire-cutting/0ce8857b-7f5f-400e-8536-6a496c724d50.avif)
![wc-2.png](../docs/images/tutorials/wire-cutting/cbce4455-4794-4c81-8630-3e3993e1b29f.avif)

Dahil ang bawat subcircuit ay may mas kaunting qubits at gates, inaasahang mas hindi sila madaling maapektuhan ng noise. Ang tutorial na ito ay nagpapakita ng halimbawa kung saan ang pamamaraang ito ay maaaring gamitin upang epektibong pigilin ang noise sa sistema.
## Mga Kinakailangan
Bago simulan ang tutorial na ito, tiyaking mayroon kayong mga sumusunod na naka-install:

- Qiskit SDK v2.0 o mas bago, na may [visualization](https://docs.quantum.ibm.com/api/qiskit/visualization) support
- Qiskit Runtime v0.22 o mas bago ( `pip install qiskit-ibm-runtime` )
- Circuit cutting Qiskit addon v0.10.0 o mas bago (`pip install qiskit-addon-cutting`)
- Qiskit addon utils 0.3 o mas bago (`pip install qiskit-addon-utils`)
- Qiskit Aer (`pip install qiskit-aer` )
## Setup

In [1]:
import numpy as np
import matplotlib.pyplot as plt

from qiskit.circuit import Parameter, ParameterVector, QuantumCircuit
from qiskit.quantum_info import PauliList, SparsePauliOp
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_aer import AerSimulator
from qiskit.result import sampled_expectation_value

from qiskit_addon_cutting.instructions import CutWire
from qiskit_addon_cutting import (
    cut_wires,
    expand_observables,
    partition_problem,
    generate_cutting_experiments,
    reconstruct_expectation_values,
)

from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import SamplerV2, Batch

## Halimbawa gamit ang maliit na simulator
Ipinapatupad ng tutorial na ito ang isang [Qiskit pattern](/guides/intro-to-patterns) upang i-simulate ang isang one-dimensional (1D) Many-Body Localization (MBL) circuit. Ang MBL circuit ay isang hardware-efficient circuit na may dalawang parameter, $\theta$ at $\vec{\phi}$. Kapag ang $\theta$ ay nakatakda sa $0$ at ang initial state ay inihanda sa $|0\rangle$ para sa lahat ng qubits, ang ideal expectation value ng $\langle Z_i \rangle$ ay $+1$ para sa bawat qubit site $i$ anuman ang mga halaga ng $\vec{\phi}$. Mas maraming detalye tungkol sa circuit na ito ay makikita sa [artikulong ito](https://www.nature.com/articles/s41467-025-57623-x).

Pansinin na sa isang noiseless simulator, ang expectation value na nakuha na may at walang circuit cutting ay magiging pareho.
### Hakbang 1: I-map ang classical inputs sa quantum problem
#### Buuin ang 1D MBL circuit
Una, nagpapakita tayo ng function para sa pagbuo ng 1D MBL circuit.

In [2]:
class MBLChainCircuit(QuantumCircuit):
    def __init__(
        self, num_qubits: int, depth: int, use_cut: bool = False
    ) -> None:
        super().__init__(
            num_qubits, name=f"MBLChainCircuit<{num_qubits}, {depth}>"
        )
        evolution = MBLChainEvolution(num_qubits, depth, use_cut)
        self.compose(evolution, inplace=True)


class MBLChainEvolution(QuantumCircuit):
    def __init__(self, num_qubits: int, depth: int, use_cut) -> None:
        super().__init__(
            num_qubits, name=f"MBLChainEvolution<{num_qubits}, {depth}>"
        )

        theta = Parameter("θ")
        phis = ParameterVector("φ", num_qubits)

        for layer in range(depth):
            layer_parity = layer % 2
            # print("layer parity", layer_parity)
            for qubit in range(layer_parity, num_qubits - 1, 2):
                # print(qubit)
                self.cz(qubit, qubit + 1)
                self.u(theta, 0, np.pi, qubit)
                self.u(theta, 0, np.pi, qubit + 1)
                if (
                    use_cut
                    and layer_parity == 0
                    and (
                        qubit == num_qubits // 2 - 1
                        or qubit == num_qubits // 2
                    )
                ):
                    self.append(CutWire(), [num_qubits // 2])
                if use_cut and layer < depth - 1 and layer_parity == 1:
                    if qubit == num_qubits // 2:
                        self.append(CutWire(), [qubit])
            for qubit in range(num_qubits):
                self.p(phis[qubit], qubit)

In [3]:
num_qubits = 10
depth = 2
mbl = MBLChainCircuit(num_qubits, depth)
mbl.draw("mpl", fold=-1)

<Image src="../docs/images/tutorials/wire-cutting/extracted-outputs/a3debf65-06df-4277-933e-14b6f6170756-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/tutorials/wire-cutting/extracted-outputs/a3debf65-06df-4277-933e-14b6f6170756-0.avif)

Kinakalkula natin ang average na expectation value na $O = \frac{1}{n} \sum_i Z_i$ sa lahat ng qubits para sa $\theta = 0$. Dahil ang ideal na expectation value ng $\langle Z_i \rangle = 1$ $\forall$ $i$, ang ideal na expectation value ng $O$ ay $1$ din. Ang mga parameter na $\phi$ ay pinili nang random.

In [4]:
np.random.seed(42)
phis = list(np.random.rand(mbl.num_parameters - 1))
theta = [0]
params = theta + phis

Ang circuit ay kailangang markahan sa pamamagitan ng paglalagay ng **CutWire** sa mga ninanais na lokasyon upang hatiin ito. Para sa tutorial na ito, pumipili tayo ng pantay na partisyon. Ang MBL circuit ay dinisenyo upang ang pagtatakda ng `use_cut=True` sa function ay wastong naglalagay ng annotation pagkatapos ng $\frac{n}{2}$ qubits, kung saan ang $n$ ay ang bilang ng mga qubits sa orihinal na circuit. Itinalaga rin natin ang mga random na nabuong parameter sa circuit.

In [5]:
mbl_cut = MBLChainCircuit(num_qubits, depth, use_cut=True)
mbl_cut.assign_parameters(params, inplace=True)
mbl_cut.draw("mpl", fold=-1)

<Image src="../docs/images/tutorials/wire-cutting/extracted-outputs/5208e0a8-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/tutorials/wire-cutting/extracted-outputs/5208e0a8-0.avif)

### Hakbang 2: I-optimize ang problema para sa quantum hardware execution
#### Putulin ang circuit sa mas maliit na subcircuits
Ngayon ay hinahati natin ang circuit sa dalawang mas maliit na subcircuits gamit ang [`qiskit-addon-cutting`](https://qiskit.github.io/qiskit-addon-cutting/). Ang `qiskit-addon-cutting` ay nagdadagdag ng virtual na `Move` gate upang hatiin ang lokasyon ng wire cut sa pamamagitan ng wastong pag-aayos ng bilang ng qubits. Ngayon ay lumilikha tayo ng circuit na may virtual na gate na ito. Dahil may isang wire cut, ang bilang ng mga nauugnay na qubits ay dadagdagan ng 1.

In [6]:
mbl_move = cut_wires(mbl_cut)
mbl_move.draw("mpl", fold=-1)

<Image src="../docs/images/tutorials/wire-cutting/extracted-outputs/1834cb22-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/tutorials/wire-cutting/extracted-outputs/1834cb22-0.avif)

#### Buuin at palawakin ang observable
Ang observable, tulad ng tinukoy dati, ay magiging average ng $Z$ sa bawat qubit. Gayunpaman, kapag nailagay na ang virtual na `Move` gate, ang epektibong bilang ng qubits sa circuit ay tumataas. Ang observable ay kailangang palawakin din nang naaayon upang isaalang-alang ang pagbabagong ito sa bilang ng mga qubits. Pansinin na ang observable ay palaging kumikilos nang trivially (bilang $I$) sa karagdagang qubit na idinagdag para sa virtual na `Move` gate.

In [7]:
observable = PauliList(
    ["I" * i + "Z" + "I" * (num_qubits - i - 1) for i in range(num_qubits)]
)
observable

PauliList(['ZIIIIIIIII', 'IZIIIIIIII', 'IIZIIIIIII', 'IIIZIIIIII',
           'IIIIZIIIII', 'IIIIIZIIII', 'IIIIIIZIII', 'IIIIIIIZII',
           'IIIIIIIIZI', 'IIIIIIIIIZ'])

In [8]:
new_obs = expand_observables(observable, mbl, mbl_move)
new_obs

PauliList(['ZIIIIIIIIII', 'IZIIIIIIIII', 'IIZIIIIIIII', 'IIIZIIIIIII',
           'IIIIZIIIIII', 'IIIIIIZIIII', 'IIIIIIIZIII', 'IIIIIIIIZII',
           'IIIIIIIIIZI', 'IIIIIIIIIIZ'])

Now the circuit can be partitioned along the `Move` gate and we obtain the subcircuits, as well as the subobservable, which is the portion of the original observable associated with each subcircuit.

In [12]:
partitioned_problem = partition_problem(circuit=mbl_move, observables=new_obs)
subcircuits = partitioned_problem.subcircuits
subobservables = partitioned_problem.subobservables

Here we visualize the two subcircuits:

In [13]:
subcircuits[0].draw("mpl", fold=-1)

<Image src="../docs/images/tutorials/wire-cutting/extracted-outputs/1b0e779f-0.avif" alt="Output of the previous code cell" />

In [14]:
subcircuits[1].draw("mpl", fold=-1)

<Image src="../docs/images/tutorials/wire-cutting/extracted-outputs/3c802f28-0.avif" alt="Output of the previous code cell" />

Dito natin biswalhin ang dalawang subcircuits:

In [15]:
M_z = SparsePauliOp(
    ["I" * i + "Z" + "I" * (num_qubits - i - 1) for i in range(num_qubits)],
    coeffs=[1 / num_qubits] * num_qubits,
)

As discussed before, for each cut the upstream circuit must be measured in a Pauli basis, and the downstream circuit must be prepared in the eigenstate of the basis. The function `generate_cutting_experiments` creates all of these necessary circuits and the coefficients associated with each circuit required for reconstruction. Find more detail in [this paper](https://journals.aps.org/prl/abstract/10.1103/PhysRevLett.125.150504).

In [16]:
subexperiments, coefficients = generate_cutting_experiments(
    circuits=subcircuits,
    observables=subobservables,
    num_samples=np.inf,
)

![Output of the previous code cell](../docs/images/tutorials/wire-cutting/extracted-outputs/3c802f28-0.avif)

Ang pagpapalawak ng observable gamit ang `Move` operation ay nangangailangan ng `PauliList` data structure. Upang ma-reconstruct ang expectation value ng orihinal na circuit, kailangan natin ang observable sa format na `SparsePauliOp`.

In [17]:
service = QiskitRuntimeService()
backend = service.least_busy(
    operational=True, simulator=False, min_num_qubits=133
)

print(backend)

<IBMBackend('ibm_fez')>


Tulad ng tinalakay dati, para sa bawat pagputol ang upstream circuit ay kailangang sukatin sa isang Pauli basis, at ang downstream circuit ay kailangang ihanda sa eigenstate ng basis. Ang function na `generate_cutting_experiments` ay lumilikha ng lahat ng kinakailangang circuits at mga coefficients na nauugnay sa bawat circuit na kinakailangan para sa reconstruction. Hanapin ang mas maraming detalye sa [papel na ito](https://journals.aps.org/prl/abstract/10.1103/PhysRevLett.125.150504).

In [20]:
pm_basis = generate_preset_pass_manager(
    optimization_level=2, basis_gates=backend.configuration().basis_gates
)
basis_subexperiments = {
    label: pm_basis.run(partition_subexpts)
    for label, partition_subexpts in subexperiments.items()
}

In [21]:
sampler = SamplerV2(mode=AerSimulator())
jobs = {
    label: sampler.run(subsystem_subexpts, shots=2**12)
    for label, subsystem_subexpts in basis_subexperiments.items()
}

### Step 4: Post-process and return result in desired classical format

Now we retrieve the result of each subexperiment run and reconstruct the expectation value of the uncut circuit:

In [22]:
# Retrieve results
results = {label: job.result() for label, job in jobs.items()}

In [23]:
reconstructed_expval_terms = reconstruct_expectation_values(
    results,
    coefficients,
    subobservables,
)
reconstructed_expval = np.dot(reconstructed_expval_terms, M_z.coeffs).real
reconstructed_expval

np.float64(0.9953821063041687)

In [32]:
methods = [
    "Uncut",
    "Wire cut",
]
values = [
    1,
    reconstructed_expval,
]  # since the ideal expectation value in noiseless simulation is +1

ax = plt.gca()
plt.bar(methods, values, color="#a56eff", width=0.4, edgecolor="#8a3ffc")
ax.set_ylabel(r"$M_Z$", fontsize=12)

Text(0, 0.5, '$M_Z$')

<Image src="../docs/images/tutorials/wire-cutting/extracted-outputs/fb5f955a-1.avif" alt="Output of the previous code cell" />

### Hakbang 4: Mag-post-process at ibalik ang resulta sa nais na classical format
Ngayon ay kinukuha natin ang resulta ng bawat subexperiment run at rine-reconstruct ang expectation value ng uncut circuit:

In [ ]:
num_qubits = 60
depth = 2

# construct the circuit
mbl = MBLChainCircuit(num_qubits, depth)

# create parameters
phis = list(np.random.rand(mbl.num_parameters - 1))
theta = [0]
params = theta + phis

# construct the cut circuit
mbl_cut = MBLChainCircuit(num_qubits, depth, use_cut=True)
mbl_cut.assign_parameters(params, inplace=True)
mbl_move = cut_wires(mbl_cut)

# Define observable and expand to account for the wire cut
observable = PauliList(
    ["I" * i + "Z" + "I" * (num_qubits - i - 1) for i in range(num_qubits)]
)
new_obs = expand_observables(observable, mbl, mbl_move)

# Construct a SparsePauliOp version of the observable for later use in reconstruction
M_z = SparsePauliOp(
    ["I" * i + "Z" + "I" * (num_qubits - i - 1) for i in range(num_qubits)],
    coeffs=[1 / num_qubits] * num_qubits,
)

# Partition the circuit and get subcircuits and subobservables
partitioned_problem = partition_problem(circuit=mbl_move, observables=new_obs)
subcircuits = partitioned_problem.subcircuits
subobservables = partitioned_problem.subobservables

# Obtain subexperiments and coefficients
subexperiments, coefficients = generate_cutting_experiments(
    circuits=subcircuits,
    observables=subobservables,
    num_samples=np.inf,
)

# Transpile the subexperiments to the backend
pm = generate_preset_pass_manager(optimization_level=2, backend=backend)
isa_subexperiments = {
    label: pm.run(partition_subexpts)
    for label, partition_subexpts in subexperiments.items()
}

# Execute the subexperiments and retrieve results
with Batch(backend=backend) as batch:
    sampler = SamplerV2(mode=batch)
    sampler.options.environment.job_tags = ["TUT_WC"]
    jobs = {
        label: sampler.run(subsystem_subexpts, shots=2**12)
        for label, subsystem_subexpts in isa_subexperiments.items()
    }
results = {label: job.result() for label, job in jobs.items()}

# Reconstruct the expectation value of the original observable
reconstructed_expval_terms = reconstruct_expectation_values(
    results,
    coefficients,
    subobservables,
)
reconstructed_expval = np.dot(reconstructed_expval_terms, M_z.coeffs).real

# Compute the uncut circuit to obtain the noisy expectation value for comparison
sampler = SamplerV2(mode=backend)
sampler.options.environment.job_tags = ["TUT_WC"]

if mbl.num_clbits == 0:
    mbl.measure_all()
isa_mbl = pm.run(mbl)

pub = (isa_mbl, params)
uncut_job = sampler.run([pub])

uncut_counts = uncut_job.result()[0].data.meas.get_counts()
uncut_expval = sampled_expectation_value(uncut_counts, M_z)

# visualize the results
ax = plt.gca()
methods = ["uncut", "cut"]
values = [uncut_expval, reconstructed_expval]

plt.bar(methods, values, color="#a56eff", width=0.4, edgecolor="#8a3ffc")
plt.axhline(y=1, color="k", linestyle="--")
plt.text(0.3, 0.95, "Exact result")
plt.show()

<Image src="../docs/images/tutorials/wire-cutting/extracted-outputs/37834c72-0.avif" alt="Output of the previous code cell" />

In [29]:
uncut_expval

0.9202473958333336

## Next steps

<Admonition type="tip" title="Recommendations">
If you found this work interesting, you might be interested in the following material:
- [Periodic boundary conditions with circuit cutting](/docs/tutorials/periodic-boundary-conditions-with-circuit-cutting)
- [Circuit cutting for depth reduction](/docs/tutorials/depth-reduction-with-circuit-cutting)
</Admonition>

## References


[1] Peng, T., Harrow, A. W., Ozols, M., & Wu, X. (2020). Simulating large quantum circuits on a small quantum computer. Physical review letters, 125(15), 150504.

[2] Tang, W., Tomesh, T., Suchara, M., Larson, J., & Martonosi, M. (2021, April). Cutqc: using small quantum computers for large quantum circuit evaluations. In Proceedings of the 26th ACM International conference on architectural support for programming languages and operating systems (pp. 473-486).

[3]  Perlin, M. A., Saleem, Z. H., Suchara, M., & Osborn, J. C. (2021). Quantum circuit cutting with maximum-likelihood tomography. npj Quantum Information, 7(1), 64.

[4]  Majumdar, R., & Wood, C. J. (2022). Error mitigated quantum circuit cutting. arXiv preprint arXiv:2211.13431.

[5]  Khare, T., Majumdar, R., Sangle, R., Ray, A., Seshadri, P. V., & Simmhan, Y. (2023). Parallelizing Quantum-Classical Workloads: Profiling the Impact of Splitting Techniques. In 2023 IEEE International Conference on Quantum Computing and Engineering (QCE) (Vol. 1, pp. 990-1000). IEEE.

[6]  Bhoumik, D., Majumdar, R., Saha, A., & Sur-Kolay, S. (2023). Distributed Scheduling of Quantum Circuits with Noise and Time Optimization. arXiv preprint arXiv:2309.06005.

[7]  Majumdar, R. (2024). Efficient Reduction of Resources and Noise in Discrete Quantum Computing Circuits (Doctoral dissertation, Indian Statistical Institute - Kolkata). https://www.proquest.com/openview/b481def90b1cc80e6b58a77c99e8385c/1?pq-origsite=gscholar&cbl=2026366&diss=y